# OpenCyc microtheories: воспроизводимый аудит

Источник — только `allAssertionsOriginal.txt` из проверенного ZIP. Зависимости: Python, pandas, NumPy, SciPy; SQLite входит в Python. Выполните все ячейки последовательно. Ноутбук находит корень workspace; вне Academia создаёт ту же структуру `user/` относительно текущей директории. Загрузка — около 46 MB, распаковка — 217 MB; SQLite и временные данные остаются в `user/user-data/tmp/cyc-microtheories/`. Для повторного запуска нужны сеть (если ZIP отсутствует) и место порядка 1 GB.

**Важно:** 900 исходных строк повреждены синтаксически. Реконструкции — только гипотезы, не исходные утверждения. Граф `genlMt` имеет циклы; DAG получается свёрткой SCC. Подсчёт переменных лексический; замкнутость Mt-термов проверяется отдельно с учётом связывателей. Ни Cyc inference engine, ни YAFOLL не запускаются.

Четыре выбранные Mt содержат следы клаузальных правил, но это не четыре проверенных исполняемых теории. Источниковые строки и целые цитаты правил позволяют проверять восстановление. Отчёт `report.md` содержит интерпретацию; численные таблицы и CycL-экспорты пересоздаются здесь.


In [1]:
from pathlib import Path
import urllib.request, zipfile, shutil, hashlib, sqlite3, json, collections, re
import sys, platform, datetime, io, contextlib
import pandas as pd
import numpy as np
import scipy
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'user/workspace').is_dir()), Path.cwd())
TMP = ROOT/'user/user-data/tmp/cyc-microtheories'
OUT = ROOT/'user/workspace/co-scientist-outputs/cyc-microtheories'
TMP.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
SOURCE = TMP/'allAssertionsOriginal.txt'
DB = TMP/'assertions.sqlite'
COMMIT = '11d96d918305569b5e836463a3f346d346e47a8e'
URL = f'https://raw.githubusercontent.com/Effervex/CycDAG/{COMMIT}/OpenCycAssertions.zip'
ZIP_SHA = '0e57a6d1a7180b5abf0acfb5943325af10c932e10bf962ae496753ac3cdf7079'
SOURCE_SHA = '27c272a75299873c921ccec8dd1956355583e015ab618c530119a85c981c9101'
archive = TMP/'OpenCycAssertions.zip'
if not archive.exists():
    urllib.request.urlretrieve(URL, archive)
assert hashlib.sha256(archive.read_bytes()).hexdigest() == ZIP_SHA
with zipfile.ZipFile(archive) as z:
    entries = [{'name': i.filename, 'bytes': i.file_size} for i in z.infolist()]
    if not SOURCE.exists():
        with z.open('allAssertionsOriginal.txt') as src, SOURCE.open('wb') as dst:
            shutil.copyfileobj(src, dst)
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == SOURCE_SHA
print('Verified archive bytes:', archive.stat().st_size)
print(entries)


Verified archive bytes: 45978030
[{'name': 'allAssertions.txt', 'bytes': 184966847}, {'name': 'allAssertionsOriginal.txt', 'bytes': 216820500}]


## Парсер и классификация

Итеративный разбор сохраняет строки и исходные токены. Категории взаимоисключающие: structural rule/context marker → верхний isa/genls → без переменных → прочее. Маркер внутри метаданных сам по себе не делает запись правилом.

In [2]:
def parse(text):
    """Iterative S-expression parser; quoted strings/escaped characters are opaque.
    Tokens keep their original spelling, including #$ and string delimiters.
    Parentheses in strings do not affect nesting. Exactly one expression required.
    """
    roots=[]; stack=[]; i=0; n=len(text)
    def add(value):
        (stack[-1] if stack else roots).append(value)
    while i<n:
        c=text[i]
        if c.isspace(): i+=1; continue
        if c==';':
            j=text.find('\n',i); i=n if j<0 else j+1; continue
        if c=='(':
            v=[]; add(v); stack.append(v); i+=1; continue
        if c==')':
            if not stack: raise ValueError('unmatched close')
            stack.pop(); i+=1; continue
        start=i
        if c in ('"','|'):
            delim=c; i+=1
            while i<n:
                if text[i]=='\\':
                    if i+1>=n: raise ValueError('dangling escape')
                    i+=2
                elif text[i]==delim: i+=1; break
                else: i+=1
            else: raise ValueError('unclosed string/symbol')
        else:
            while i<n and not text[i].isspace() and text[i] not in '()': i+=1
        add(text[start:i])
    if stack: raise ValueError('unclosed list')
    if len(roots)!=1: raise ValueError('expected one expression')
    return roots[0]

def cyc(x): return '('+' '.join(map(cyc,x))+')' if isinstance(x,list) else x

def inspect(x):
    heads=set(); variable=False; todo=[x]
    while todo:
        v=todo.pop()
        if isinstance(v,list):
            if v and isinstance(v[0],str): heads.add(v[0])
            todo.extend(v)
        elif v.startswith('?'): variable=True
    return heads,variable

def build_db():
    if DB.exists(): DB.unlink()
    con=sqlite3.connect(DB)
    con.executescript('PRAGMA journal_mode=OFF; PRAGMA synchronous=OFF; CREATE TABLE assertions(line INTEGER PRIMARY KEY, formula TEXT, mt TEXT, predicate TEXT, category TEXT, ground INTEGER, implication INTEGER, expansion INTEGER, ist INTEGER); CREATE TABLE parse_errors(line INTEGER, error TEXT, raw TEXT); CREATE TABLE edges(child TEXT,parent TEXT,asserted_in TEXT,line INTEGER);')
    batch=[]; errors=[]; edges=[]; total=0; mt_empty=0; nonstandard=[]
    with SOURCE.open(encoding='utf-8',errors='strict') as f:
        for ln,raw in enumerate(f,1):
            total=ln
            try:
                text=raw.rstrip('\r\n')
                formula,sep,mt=text.rpartition('\t')
                if not sep: raise ValueError('missing TAB separator')
                ast=parse(formula)
                if not isinstance(ast,list) or not ast: raise ValueError('not a nonempty formula')
                mt=mt.strip()
                if mt: mt=cyc(parse(mt))
                else: mt_empty+=1
                pred=cyc(ast[0]); heads,var=inspect(ast)
                imp=int('#$implies' in heads); exp=int('#$expansion' in heads); ist=int('#$ist' in heads)
                category='rule_or_context' if imp or exp or ist else 'taxonomy' if pred in ('#$isa','#$genls') else 'ground_fact' if not var else 'other_non_ground'
                batch.append((ln,formula,mt,pred,category,int(not var),imp,exp,ist))
                if pred=='#$genlMt':
                    if len(ast)==3: edges.append((cyc(ast[1]),cyc(ast[2]),mt,ln))
                    else: nonstandard.append(ln)
            except Exception as e: errors.append((ln,str(e),raw.rstrip('\r\n')))
            if ln%10000==0:
                con.executemany('INSERT INTO assertions VALUES(?,?,?,?,?,?,?,?,?)',batch); batch=[]
                con.commit()
    con.executemany('INSERT INTO assertions VALUES(?,?,?,?,?,?,?,?,?)',batch)
    con.executemany('INSERT INTO parse_errors VALUES(?,?,?)',errors)
    con.executemany('INSERT INTO edges VALUES(?,?,?,?)',edges)
    con.executescript('CREATE INDEX mt_idx ON assertions(mt); CREATE INDEX pred_idx ON assertions(predicate);')
    con.commit()
    summary={'total_lines':total,'parsed':con.execute('select count(*) from assertions').fetchone()[0],'parse_errors':len(errors),'empty_mt':mt_empty,'nonstandard_genlMt_lines':nonstandard,'sha256_source':hashlib.sha256(SOURCE.read_bytes()).hexdigest(),'sha256_zip':hashlib.sha256((TMP/'OpenCycAssertions.zip').read_bytes()).hexdigest()}
    (TMP/'ingest.json').write_text(json.dumps(summary,indent=2))
    print(json.dumps(summary)); print('ERROR SAMPLE',errors[:8]); return con



In [3]:
SELECTED=['#$FinancialTransactionMt','#$MathMt','#$SetTheoryMt','#$CoreCycLMt']

def vars_in(x):
    if isinstance(x,str): return {x} if x.startswith('?') else set()
    return set().union(*(vars_in(a) for a in x)) if x else set()

BINDERS={'#$Kappa','#$Lambda','#$thereExists','#$forAll','#$TheCollectionOf','#$TheSetOf','#$TheUnique','#$TheFn','#$The'}
def free_vars(x,bound=frozenset()):
    if isinstance(x,str): return ({x}-bound) if x.startswith('?') else set()
    if not x: return set()
    if isinstance(x[0],str) and x[0] in BINDERS and len(x)>=3:
        b=bound|vars_in(x[1]); return set().union(*(free_vars(a,b) for a in x[2:]))
    return set().union(*(free_vars(a,bound) for a in x))

def repair_candidate(f):
    """Structural hypotheses only: never treated as asserted source formulas."""
    try:
        x=parse('(('+f+'))')
        if len(x)==2 and all(isinstance(a,list) for a in x) and all(isinstance(a,list) and a for side in x for a in side):
            return 'add_two_parens_each_end',x
    except (ValueError,TypeError): pass
    if f.endswith(') NI'):
        try:
            x=parse('(('+f[:-2]+'NIL)')
            if len(x)==2 and isinstance(x[0],list) and x[1]=='NIL':
                return 'add_prefix_restore_NIL',x
        except ValueError: pass
    return 'unresolved',None

def candidate_implication(x):
    def connective(head,items,empty):
        if not items: return empty
        return items[0] if len(items)==1 else [head,*items]
    antecedent=connective('#$and',x[0],'#$True')
    positive=[] if x[1]=='NIL' else x[1]
    consequent=connective('#$or',positive,'#$False')
    return ['#$implies',antecedent,consequent]

def summarize():
    con=sqlite3.connect(DB)
    con.executescript('DROP TABLE IF EXISTS rule_candidates; CREATE TABLE rule_candidates(line INTEGER PRIMARY KEY, mt TEXT, repair TEXT, raw_formula TEXT, candidate_clause TEXT, candidate_implication TEXT);')
    errors=[]
    for ln,error,raw in con.execute('select * from parse_errors').fetchall():
        f,_,mt=raw.rpartition('\t');mt=cyc(parse(mt)) if mt else ''
        mode,x=repair_candidate(f)
        clause=cyc(x) if x else '';imp=cyc(candidate_implication(x)) if x else ''
        con.execute('INSERT INTO rule_candidates VALUES(?,?,?,?,?,?)',(ln,mt,mode,f,clause,imp))
        errors.append((ln,mt,error,mode,f,clause,imp))
    con.commit()
    targets={r[5] for r in errors if r[5]};matches=collections.defaultdict(list)
    for ln,formula,mt,pred in con.execute('select line,formula,mt,predicate from assertions where ground=0 or implication+expansion+ist>0'):
        todo=[parse(formula)]
        while todo:
            v=todo.pop()
            if not isinstance(v,list):continue
            if len(v)==2 and isinstance(v[0],list) and (isinstance(v[1],list) or v[1]=='NIL'):
                key=cyc(v)
                if key in targets:matches[key].append({'source_line':ln,'mt':mt,'predicate':pred})
            todo.extend(a for a in v if isinstance(a,list))
    audit=pd.DataFrame(errors,columns=['source_line','mt','parse_error','repair_hypothesis','raw_formula','candidate_clause','candidate_implication'])
    audit['intact_quote_matches']=audit.candidate_clause.map(lambda key:json.dumps(matches.get(key,[]),ensure_ascii=False))
    audit['intact_quote_match_count']=audit.candidate_clause.map(lambda key:len(matches.get(key,[])))
    audit.to_csv(OUT/'parse_audit.csv',index=False,encoding='utf-8-sig')

    # Closed non-atomic Mt terms are kept; schema edges with free variables excluded.
    edge_rows=[];excluded=[]
    for child,parent,mt,ln in con.execute('select * from edges'):
        if free_vars(parse(child)) or free_vars(parse(parent)):
            excluded.append((child,parent,mt,ln));continue
        edge_rows.append((child,parent,mt,ln))
    edf=pd.DataFrame(edge_rows,columns=['child_mt','parent_mt','asserted_in_mt','source_line'])
    edf.to_csv(OUT/'mt_hierarchy.csv',index=False,encoding='utf-8-sig')
    labels={r[0] for r in con.execute('select distinct mt from assertions') if r[0]}
    labels.update(r[1] for r in errors if r[1])
    nodes=sorted(labels|set(edf.child_mt)|set(edf.parent_mt));ix={n:i for i,n in enumerate(nodes)}
    parents={n:set() for n in nodes};children={n:set() for n in nodes}
    for child,parent,_,_ in edge_rows: parents[child].add(parent);children[parent].add(child)
    unique_edges={(a,b) for a,b,_,_ in edge_rows}
    mat=csr_matrix((np.ones(len(unique_edges)),([ix[a] for a,b in unique_edges],[ix[b] for a,b in unique_edges])),shape=(len(nodes),len(nodes)))
    nc,lab=connected_components(mat,directed=True,connection='strong')
    comps=collections.defaultdict(list)
    for n,l in zip(nodes,lab):comps[int(l)].append(n)
    cycles=[v for v in comps.values() if len(v)>1 or (v[0],v[0]) in unique_edges]
    cp={k:set() for k in comps};cc={k:set() for k in comps}
    for a,b in unique_edges:
        ca,cb=int(lab[ix[a]]),int(lab[ix[b]])
        if ca!=cb:cp[ca].add(cb);cc[cb].add(ca)
    queue=collections.deque(k for k in comps if not cp[k]);todo={k:len(v) for k,v in cp.items()};order=[]
    while queue:
        k=queue.popleft();order.append(k)
        for ch in cc[k]:
            todo[ch]-=1
            if todo[ch]==0:queue.append(ch)
    assert len(order)==len(comps)
    condensation_roots=[k for k in comps if not cp[k]]
    condensed_edges=sorted((k,p) for k in cp for p in cp[k])
    pd.DataFrame(condensed_edges,columns=['child_scc','parent_scc']).to_csv(OUT/'mt_condensation.csv',index=False,encoding='utf-8-sig')
    anc={};desc={};dmin={};dmax={}
    for k in order:
        anc[k]=set().union(*(anc[p]|{p} for p in cp[k])) if cp[k] else set()
        dmin[k]=min(dmin[p]+1 for p in cp[k]) if cp[k] else 0
        dmax[k]=max(dmax[p]+1 for p in cp[k]) if cp[k] else 0
    for k in reversed(order):desc[k]=set().union(*(desc[ch]|{ch} for ch in cc[k])) if cc[k] else set()
    def expanded(k,which):return set().union(*(set(comps[a]) for a in which[k])) if which[k] else set()
    ancestors={n:expanded(int(lab[ix[n]]),anc)|(set(comps[int(lab[ix[n]] )])-{n}) for n in nodes}
    descendants={n:expanded(int(lab[ix[n]]),desc)|(set(comps[int(lab[ix[n]] )])-{n}) for n in nodes}
    counts=pd.read_sql_query('select mt,count(*) own_parsed,sum(ground) lexical_ground,sum(implication) nested_implies,sum(expansion) nested_expansion,sum(ist) nested_ist from assertions group by mt',con).set_index('mt').to_dict('index')
    cats=pd.read_sql_query('select mt,category,count(*) n from assertions group by mt,category',con).pivot(index='mt',columns='category',values='n').fillna(0).astype(int).to_dict('index')
    malformed=collections.Counter(e[1] for e in errors)
    repairable=collections.Counter(e[1] for e in errors if e[3]!='unresolved')
    roots=[n for n in nodes if not parents[n]]
    stats=[]
    for n in nodes:
        k=int(lab[ix[n]]);own=counts.get(n,{})
        r={'mt':n,'has_own_assertions':n in labels,'own_total':own.get('own_parsed',0)+malformed[n],**{key:own.get(key,0) for key in ['own_parsed','lexical_ground','nested_implies','nested_expansion','nested_ist']},**{key:cats.get(n,{}).get(key,0) for key in ['ground_fact','taxonomy','rule_or_context','other_non_ground']},'malformed':malformed[n],'repairable_rule_candidates':repairable[n],'parent_count':len(parents[n]),'child_count':len(children[n]),'ancestor_count':len(ancestors[n]),'descendant_count':len(descendants[n]),'min_root_distance':dmin[k],'max_root_distance':dmax[k],'scc':k,'scc_size':len(comps[k]),'scc_is_root':not cp[k],'scc_is_leaf':not cc[k],'is_root':not parents[n],'is_leaf':not children[n],'inherited_source_rows':sum(counts.get(a,{}).get('own_parsed',0)+malformed[a] for a in ancestors[n])}
        stats.append(r)
    sdf=pd.DataFrame(stats).sort_values('mt');sdf.to_csv(OUT/'mt_statistics.csv',index=False,encoding='utf-8-sig')
    pdf=pd.read_sql_query('select mt,predicate,category,count(*) assertion_count,sum(ground) lexical_ground_count,sum(implication) nested_implies_count,sum(expansion) nested_expansion_count,sum(ist) nested_ist_count from assertions group by mt,predicate,category order by mt,predicate,category',con)
    pdf.to_csv(OUT/'mt_predicate_counts.csv',index=False,encoding='utf-8-sig')
    # Own rows are fully preserved. Ancestors are listed, not silently materialized.
    selected=[]
    for mt in SELECTED:
        a=sorted(ancestors[mt]);filename=mt[2:]+'.cycl'
        with (OUT/filename).open('w',encoding='utf-8-sig') as f:
            f.write(';; Own assertions with explicit source Mt; original formulas are unchanged.\n')
            f.write(';; No Cyc engine or logical inference has been run.\n')
            f.write(';; Ancestors below are computed from all closed genlMt edges, irrespective of assertion Mt.\n')
            f.write(';; Ancestor assertions are NOT duplicated here; see notebook export_inherited().\n')
            f.write(';; Selected Mt: '+mt+'\n;; Reachable ancestor targets excluding self; SCC peers included ('+str(len(a))+'):\n')
            for n in a:f.write(';; ancestor: '+n+'\n')
            for ln,formula in con.execute('select line,formula from assertions where mt=? order by line',(mt,)):
                f.write(f';; source_line: {ln}\n(#$ist {mt} {formula})\n')
            for ln,mode,raw,clause,imp in con.execute('select line,repair,raw_formula,candidate_clause,candidate_implication from rule_candidates where mt=? order by line',(mt,)):
                f.write(f';; MALFORMED source_line: {ln}; hypothesis: {mode}\n;; RAW: {raw}\n')
                if clause:
                    f.write(';; CANDIDATE CLAUSE (NOT ASSERTED): '+clause+'\n;; CANDIDATE IMPLICATION (NOT ASSERTED): '+imp+'\n')
                    for match in matches.get(clause,[]):f.write(';; INTACT QUOTED STRUCTURE: '+json.dumps(match)+'\n')
        selected.append(sdf[sdf.mt==mt].iloc[0].to_dict()|{'ancestors':a,'file':filename})
    # Full inherited rows for selected Mts in one deduplicated, compressed provenance-preserving file.
    import gzip
    inherited_union=sorted(set().union(*(ancestors[n] for n in SELECTED)))
    with gzip.open(OUT/'selected_ancestor_assertions.cycl.gz','wt',encoding='utf-8-sig') as f:
        f.write(';; Shared ancestor own-assertion catalog. Pair with ancestor lists in each Mt .cycl.\n;; Each parsed row is wrapped in ist with its ORIGINAL Mt; malformed rows remain comments.\n')
        for mt in inherited_union:
            f.write(';; BEGIN SOURCE MT '+mt+'\n')
            for ln,formula in con.execute('select line,formula from assertions where mt=? order by line',(mt,)):
                f.write(f';; source_line: {ln}\n(#$ist {mt} {formula})\n')
            for ln,raw in con.execute('select line,raw_formula from rule_candidates where mt=? order by line',(mt,)):
                f.write(f';; MALFORMED source_line: {ln}\n;; RAW: {raw}\n')
    longest_node=max(nodes,key=lambda n:dmax[int(lab[ix[n]])]);chain=[longest_node];k=int(lab[ix[longest_node]])
    while cp[k]:
        k=max(cp[k],key=lambda p:dmax[p]);chain.append(comps[k][0])
    summary=json.loads((TMP/'ingest.json').read_text())|{'node_count':len(nodes),'assertion_mt_labels':len(labels),'edge_rows':len(edge_rows),'unique_edges':len(unique_edges),'excluded_open_schema_edges':excluded,'edge_contexts':collections.Counter(r[2] for r in edge_rows),'scc_count':nc,'condensation_edges':len(condensed_edges),'condensation_root_count':len(condensation_roots),'nonisolated_root_components':[comps[k] for k in condensation_roots if cc[k]],'isolated_nodes':sum(not parents[n] and not children[n] for n in nodes),'self_loop_edges':sum(a==b for a,b in unique_edges),'multi_node_sccs':sum(len(v)>1 for v in comps.values()),'cycles':cycles,'max_depth_condensation':max(dmax.values()),'longest_chain':chain,'roots':roots,'leaves':sum(not children[n] for n in nodes),'root_with_edges_count':sum(bool(children[n]) for n in roots),'selected':selected,'repair_hypotheses':collections.Counter(e[3] for e in errors),'candidate_records_with_intact_quotes':sum(bool(matches.get(e[5])) for e in errors),'unique_candidate_clauses':len(targets),'unique_candidate_clauses_with_intact_quotes':len(matches),'categories':dict(con.execute('select category,count(*) from assertions group by category')),'top_level_rule_heads':dict(con.execute("select predicate,count(*) from assertions where predicate in ('#$implies','#$ist','#$expansion','#$trueRule') group by predicate")),'global_predicates':dict(con.execute('select predicate,count(*) from assertions group by predicate order by count(*) desc limit 20')),'ancestor_catalog_mts':inherited_union,'ancestor_catalog_source_rows':sum(counts.get(a,{}).get('own_parsed',0)+malformed[a] for a in inherited_union),'unique_assertion_mt_pairs':con.execute('select count(*) from (select distinct formula,mt from assertions)').fetchone()[0]}
    (TMP/'summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False,default=int))
    print(json.dumps({k:v for k,v in summary.items() if k not in ['roots','selected','global_predicates']},indent=2,default=int))
    print('ROOT SAMPLE',roots[:30]);print('TOP DESC',sdf.sort_values('descendant_count',ascending=False)[['mt','descendant_count','child_count']].head(12).to_string(index=False));print('TOP ANC',sdf.sort_values(['ancestor_count','max_root_distance'],ascending=False)[['mt','ancestor_count','max_root_distance']].head(12).to_string(index=False));print('SELECTED',json.dumps(selected,indent=2,default=int))
    return summary



In [4]:
# Parentheses, quotes, escaped strings, tabs, Unicode, nested lists and dotted tails.
x = parse(r'(#$p (#$f ?X) "a (b) \"quoted\" c")')
assert x[1] == ['#$f', '?X']
assert len(parse('(#$p "a\tb" "текст")')) == 3
assert parse('(#$f . ?ARGS)') == ['#$f', '.', '?ARGS']
assert parse(cyc(x)) == x
for bad in ['(#$p', '(#$p))', '(a) (b)', '(a "unfinished)']:
    try: parse(bad)
    except ValueError: pass
    else: raise AssertionError(bad)
assert not free_vars(parse('(#$Kappa (?X) (#$p ?X))'))
assert free_vars(parse('(#$MtSpace . ?X)')) == {'?X'}
mode, clause = repair_candidate('(#$p ?X)) ((#$q ?X) (#$r ?X)')
assert mode == 'add_two_parens_each_end'
assert candidate_implication(clause)[2][0] == '#$or'  # Not a conjunction of heads.
assert repair_candidate('(#$p ?X)) NI')[1][1] == 'NIL'
assert repair_candidate('(#$p ?X) (#$q ?X)')[0] == 'unresolved'
print('Parser, binder and conservative-repair tests passed.')


Parser, binder and conservative-repair tests passed.


## SQLite и pandas

Таблица assertions содержит только строго разобранные формулы, parse_errors — все остальные строки. Гипотезы ремонта хранятся отдельно в rule_candidates. Повторный запуск пересоздаёт временную SQLite-базу.

In [5]:
log = io.StringIO()
with contextlib.redirect_stdout(log):
    con = build_db()
print(log.getvalue()[:1800])
assert con.execute('SELECT COUNT(*) FROM assertions').fetchone()[0] == 1888976
assert con.execute('SELECT COUNT(*) FROM parse_errors').fetchone()[0] == 900
con.close()


{"total_lines": 1889876, "parsed": 1888976, "parse_errors": 900, "empty_mt": 0, "nonstandard_genlMt_lines": [], "sha256_source": "27c272a75299873c921ccec8dd1956355583e015ab618c530119a85c981c9101", "sha256_zip": "0e57a6d1a7180b5abf0acfb5943325af10c932e10bf962ae496753ac3cdf7079"}
ERROR SAMPLE [(48371, 'unmatched close', '(#$genls ?var0 ?var1) (#$defnSufficient ?var0 ?var2)) ((#$defnSufficient ?var1 ?var2)\t#$BaseKB'), (48372, 'unmatched close', '(#$genls ?var0 ?var1)) ((#$genls ?var1 #$Thing)\t#$BaseKB'), (48381, 'unmatched close', '(#$disjointWith ?var0 ?var1) (#$genls ?var2 ?var1)) ((#$disjointWith ?var0 ?var2)\t#$BaseKB'), (48407, 'unmatched close', '(#$greaterThanOrEqualTo ?var0 ?var1) (#$maxQuantValue ?var2 (?var3 ?var0)) (#$minQuantValue ?var2 (?var3 ?var1))) ((#$equals ?var2 (?var3 ?var1 ?var0))\t#$BaseKB'), (48408, 'unmatched close', '(#$maxQuantValue ?var0 (?var1 ?var2)) (#$minQuantValue ?var0 (?var1 ?var2))) ((#$equals ?var0 (?var1 ?var2))\t#$BaseKB'), (48475, 'unmatched close'

## Граф, SCC и экспорты

Ребро: специализированная Mt → более общая Mt. Один шаблон с открытой переменной исключён; 14 замкнутых Kappa-термов сохранены. Восстановленные правила не добавляют рёбер. Повторные пути не размножают наследуемые строки; исходные дубликаты сохраняются. Все контексты утверждения ребра остаются в mt_hierarchy.csv.

In [6]:
log = io.StringIO()
with contextlib.redirect_stdout(log):
    result = summarize()
assert result['node_count'] == 13290
assert result['unique_edges'] == 25106
assert result['scc_count'] == 13128
assert result['max_depth_condensation'] == 87
print({k: result[k] for k in ['total_lines', 'parsed', 'parse_errors', 'node_count', 'unique_edges', 'scc_count', 'max_depth_condensation', 'candidate_records_with_intact_quotes']})


{'total_lines': 1889876, 'parsed': 1888976, 'parse_errors': 900, 'node_count': 13290, 'unique_edges': 25106, 'scc_count': 13128, 'max_depth_condensation': 87, 'candidate_records_with_intact_quotes': 158}


In [7]:
con = sqlite3.connect(DB)
stats = pd.read_csv(OUT/'mt_statistics.csv', encoding='utf-8-sig')
print('SELECTED MTS')
print(stats[stats.mt.isin(SELECTED)][['mt','own_total','own_parsed','malformed','repairable_rule_candidates','ancestor_count']].to_string(index=False))
print('MOST INHERITED (reachability, not logical inference)')
print(stats.sort_values(['descendant_count','mt'], ascending=[False,True])[['mt','descendant_count']].head(10).to_string(index=False))
print('MOST ANCESTORS')
print(stats.sort_values(['ancestor_count','mt'], ascending=[False,True])[['mt','ancestor_count','max_root_distance']].head(10).to_string(index=False))
print('ROOT COMPONENTS WITH EDGES:', result['nonisolated_root_components'])
print('PREDICATES FOR FINANCIAL TRANSACTIONS')
print(pd.read_sql_query('SELECT predicate, category, COUNT(*) n FROM assertions WHERE mt=? GROUP BY predicate,category ORDER BY n DESC', con, params=['#$FinancialTransactionMt']).head(15).to_string(index=False))


SELECTED MTS
                      mt  own_total  own_parsed  malformed  repairable_rule_candidates  ancestor_count
            #$CoreCycLMt        548         526         22                          18               3
#$FinancialTransactionMt        152         146          6                           6             162
                #$MathMt         22           1         21                          21              10
           #$SetTheoryMt          4           2          2                           2               9
MOST INHERITED (reachability, not logical inference)
                                   mt  descendant_count
       #$LogicalTruthImplementationMt              8649
                     #$LogicalTruthMt              8649
           #$CoreCycLImplementationMt              8647
                         #$CoreCycLMt              8647
                             #$BaseKB              8645
                  #$CycAgencyTheoryMt              8645
   #$CycHistoricalPossibili

In [8]:
def export_inherited(mt, destination):
    """Materialize reachable ancestor source rows, not deductions. Never assert repairs."""
    import gzip
    selected = next(item for item in result['selected'] if item['mt'] == mt)
    op = gzip.open if str(destination).endswith('.gz') else open
    with sqlite3.connect(DB) as export_con, op(destination, 'wt', encoding='utf-8-sig') as out:
        for parent in selected['ancestors']:
            for ln, formula in export_con.execute('SELECT line,formula FROM assertions WHERE mt=? ORDER BY line', (parent,)):
                out.write(f';; source_line: {ln}\n(#$ist {parent} {formula})\n')
            for ln, raw in export_con.execute('SELECT line,raw_formula FROM rule_candidates WHERE mt=? ORDER BY line', (parent,)):
                out.write(f';; MALFORMED source_line: {ln}\n;; RAW: {raw}\n')
# Optional: export_inherited('#$MathMt', OUT/'MathMt_inherited.cycl.gz')
# The shared selected_ancestor_assertions.cycl.gz already contains the union,
# keeping each source record once and preserving its actual assertion Mt.
# Raw original bytes can always be retrieved by source_line from SOURCE.


## Проверки и происхождение

CycL-файлы содержат собственные формулы с добавленной оболочкой ist, перечень достижимых предков и повреждённые строки в комментариях. Добавленные ist — способ представления экспорта, не новые ist из исходного дампа. Общий gzip-каталог содержит полные собственные записи всех предков выбранных Mt, с исходной Mt и номером строки.

In [9]:
# Export integrity: every parsed own record is emitted once; malformed rows are comments.
for item in result['selected']:
    lines = (OUT/item['file']).read_text(encoding='utf-8-sig').splitlines()
    forms = [line for line in lines if line.startswith('(#$ist ')]
    assert len(forms) == item['own_parsed']
    assert sum(line.startswith(';; MALFORMED source_line:') for line in lines) == item['malformed']
    for line in forms:
        ast = parse(line)
        assert ast[0] == '#$ist' and ast[1] == item['mt']
assert sum(result['categories'].values()) + result['parse_errors'] == result['total_lines']
assert not result['nonstandard_genlMt_lines']
assert len(result['excluded_open_schema_edges']) == 1
# Validate the complete shared ancestor catalog, not just each small own export.
import gzip
source_ids = set(); parsed_catalog = malformed_catalog = 0
with gzip.open(OUT/'selected_ancestor_assertions.cycl.gz', 'rt', encoding='utf-8-sig') as catalog:
    for line in catalog:
        if line.startswith(';; source_line: '):
            source_id = int(line.split(':',1)[1]); assert source_id not in source_ids
            source_ids.add(source_id)
        elif line.startswith(';; MALFORMED source_line: '):
            source_id = int(line.split(':',1)[1]); assert source_id not in source_ids
            source_ids.add(source_id); malformed_catalog += 1
        elif line.startswith('(#$ist '):
            assert parse(line)[0] == '#$ist'; parsed_catalog += 1
assert len(source_ids) == parsed_catalog + malformed_catalog == result['ancestor_catalog_source_rows']
export_inherited('#$CoreCycLMt', TMP/'helper_test.cycl.gz')
with gzip.open(TMP/'helper_test.cycl.gz', 'rt', encoding='utf-8-sig') as test:
    assert sum(line.startswith(';; source_line: ') or line.startswith(';; MALFORMED source_line: ') for line in test) == 431
(TMP/'helper_test.cycl.gz').unlink()
# SCC peers are reachable ancestors, so do not interpret ancestor_count as a strict order.
print('Row accounting, exported formulas, inherited catalog and hierarchy checks passed.')
provenance = {
    'analysis_timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'source_url_supplied': 'https://raw.githubusercontent.com/Effervex/CycDAG/master/OpenCycAssertions.zip',
    'pinned_url': URL, 'repository_commit': COMMIT,
    'git_blob_sha1': '4f360aca565a403e09669c5aa5ec23cbc86d6f0d',
    'zip_sha256': ZIP_SHA, 'source_sha256': SOURCE_SHA,
    'zip_bytes': archive.stat().st_size, 'zip_entries': entries,
    'release_identification': 'Likely OpenCyc 4.0; exact export version unverified',
    'attribution': 'OpenCyc KB: Cycorp; CycDAG repository: Sam Sarjant / University of Waikato',
    'licensing': {'kb_content': 'CC-BY-3.0 stated by requester; not independently confirmed for this ZIP', 'repository': 'GPL-3.0 per GitHub metadata; do not conflate code and KB licences'},
    'versions': {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'scipy': scipy.__version__, 'sqlite': sqlite3.sqlite_version},
    'sqlite_schema': list(con.execute("SELECT sql FROM sqlite_master WHERE type='table'")),
    'metrics': {k:v for k,v in result.items() if k not in ['roots','cycles']},
    'semantics': {'edge_direction': 'child/specialized -> parent/general', 'graph_scope': 'union of asserted ground-or-closed genlMt edges across assertion Mts', 'depth': 'longest path from a parentless SCC in condensation DAG; edge count', 'ancestor_count': 'distinct reachable targets excluding self, including SCC peers', 'inherited_content': 'computed ancestor-source union, not stored or inferred conclusions', 'ground': 'lexical no-variable test for counts; binder-aware free-variable test only for hierarchy terms', 'repairs': 'hypotheses retained as comments, never counted as parsed source assertions'},
}
(OUT/'provenance.json').write_text(json.dumps(provenance, ensure_ascii=False, indent=2, default=int), encoding='utf-8')
print('Provenance written. SQLite remains in the temporary directory:', DB.relative_to(ROOT))
con.close()


Row accounting, exported formulas, inherited catalog and hierarchy checks passed.
Provenance written. SQLite remains in the temporary directory: user/user-data/tmp/cyc-microtheories/assertions.sqlite
